# Lumpy 04: Phase 3 Trend + Agent Review

Notebook version: 3

Phase 3 reads the saved Phase 2 outputs. It does not train models.

It checks whether the chosen model is:

- getting the demand level right
- following the month-to-month trend/shape
- still acceptable on SKU-horizon allocation


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


NOTEBOOK_VERSION = 3
REQUIRED_WINDOW_MONTHS = 18
REQUIRED_LAG_MONTHS = 3


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [
        start,
        start.parent,
        start.parent.parent,
        start / "Inchscape Repo",
        start.parent / "Inchscape Repo",
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "results").exists():
            return candidate.resolve()
    raise FileNotFoundError("Open this notebook from the Inchscape Repo folder or update find_repo_root().")


def read_table(filename, *, parse_dates=None, required=False):
    path = table_dir / filename
    if path.exists():
        return pd.read_csv(path, parse_dates=parse_dates)
    if required:
        raise FileNotFoundError(f"Missing required lumpy output: {path}")
    return pd.DataFrame()


root = find_repo_root()
table_dir = root / "results" / "lumpy_outputs" / "tables"

REPORT_FILES = {
    "metric_decision": "lumpy_agent_metric_decision_report.csv",
    "lag_comparison": "lumpy_agent_lag_comparison_report.csv",
    "monthly_total": "lumpy_agent_monthly_total_report.csv",
    "sku_horizon": "lumpy_agent_sku_horizon_report.csv",
    "population_strategy": "lumpy_agent_population_strategy_report.csv",
    "segment_strategy": "lumpy_agent_segment_strategy_recommendations.csv",
}

reports = {name: read_table(filename) for name, filename in REPORT_FILES.items()}
monthly_totals = read_table("lumpy_monthly_total_results.csv", parse_dates=["month"], required=True)
metric_suite = read_table("lumpy_metric_suite_model_summary.csv", required=True)
monthly_total_model_summary = read_table("lumpy_monthly_total_model_summary.csv", required=True)
sku_horizon_model_summary = read_table("lumpy_sku_horizon_model_summary.csv", required=True)

required_metric = metric_suite.loc[
    metric_suite["test_months"].eq(REQUIRED_WINDOW_MONTHS)
    & metric_suite["gap_months"].eq(REQUIRED_LAG_MONTHS)
].copy()
lead_model = (
    required_metric.sort_values(["phase1_decision_score_percent", "model"]).iloc[0]["model"]
    if not required_metric.empty
    else pd.NA
)

print(f"Notebook version: {NOTEBOOK_VERSION}")
print(f"Project root: {root}")
print("Loaded reports:", [name for name, report in reports.items() if not report.empty])
print(f"Required benchmark: {REQUIRED_WINDOW_MONTHS}m window / {REQUIRED_LAG_MONTHS}m lag")
print(f"Lead model: {lead_model}")


## Phase 3 Trend Shape Report


In [ ]:
def linear_slope(values):
    values = pd.Series(values).astype(float).to_numpy()
    if len(values) < 2 or np.nanstd(values) == 0:
        return 0.0
    return float(np.polyfit(np.arange(len(values), dtype=float), values, 1)[0])


def trend_status(actual_slope, forecast_slope):
    if pd.isna(actual_slope) or pd.isna(forecast_slope):
        return "not_enough_data"
    if abs(actual_slope) < 1.0:
        return "actual_is_flat"
    flat_threshold = max(abs(actual_slope) * 0.25, 1.0)
    if abs(forecast_slope) < flat_threshold:
        return "flat_forecast_vs_actual_trend"
    if np.sign(actual_slope) != np.sign(forecast_slope):
        return "wrong_direction"
    ratio = forecast_slope / actual_slope if abs(actual_slope) > 1e-9 else np.nan
    if pd.notna(ratio) and 0.5 <= ratio <= 1.5:
        return "trend_direction_and_scale_captured"
    return "trend_direction_present_scale_off"


def build_trend_shape_report(monthly_totals):
    data = monthly_totals.copy()
    data["month"] = pd.to_datetime(data["month"], errors="coerce")
    data["actual"] = pd.to_numeric(data["actual"], errors="coerce").fillna(0.0)
    data["forecast"] = pd.to_numeric(data["forecast"], errors="coerce").fillna(0.0)
    group_columns = [
        column
        for column in ["window_label", "test_months", "gap_months", "fold_id", "global_fold_id", "model"]
        if column in data.columns
    ]

    rows = []
    for keys, group in data.groupby(group_columns, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        key_map = dict(zip(group_columns, keys))
        group = group.sort_values("month")
        actual = group["actual"]
        forecast = group["forecast"]
        actual_total = float(actual.sum())
        forecast_total = float(forecast.sum())
        absolute_error = float((actual - forecast).abs().sum())
        actual_slope = linear_slope(actual)
        forecast_slope = linear_slope(forecast)
        actual_std = float(actual.std(ddof=0))
        forecast_std = float(forecast.std(ddof=0))
        correlation = (
            float(np.corrcoef(actual, forecast)[0, 1])
            if actual_std > 0 and forecast_std > 0
            else np.nan
        )
        rows.append(
            {
                **key_map,
                "months": int(group["month"].nunique()),
                "actual_total": actual_total,
                "forecast_total": forecast_total,
                "monthly_total_wmape_percent": 100 * absolute_error / actual_total if actual_total > 0 else np.nan,
                "mean_rolling_3m_wmape_percent": float(group["rolling_3m_wmape_percent"].mean())
                if "rolling_3m_wmape_percent" in group.columns
                else np.nan,
                "actual_slope_per_month": actual_slope,
                "forecast_slope_per_month": forecast_slope,
                "slope_gap_per_month": forecast_slope - actual_slope,
                "slope_capture_ratio": forecast_slope / actual_slope if abs(actual_slope) > 1e-9 else np.nan,
                "actual_cv_percent": 100 * actual_std / actual.mean() if actual.mean() > 0 else np.nan,
                "forecast_cv_percent": 100 * forecast_std / forecast.mean() if forecast.mean() > 0 else np.nan,
                "actual_forecast_correlation": correlation,
                "trend_status": trend_status(actual_slope, forecast_slope),
            }
        )
    return pd.DataFrame(rows)


trend_shape_report = build_trend_shape_report(monthly_totals)
fold_id_column = "global_fold_id" if "global_fold_id" in trend_shape_report.columns else "fold_id"
required_trend = trend_shape_report.loc[
    trend_shape_report["test_months"].eq(REQUIRED_WINDOW_MONTHS)
    & trend_shape_report["gap_months"].eq(REQUIRED_LAG_MONTHS)
].copy()
latest_required_trend = required_trend.loc[
    required_trend[fold_id_column].eq(required_trend[fold_id_column].max())
].copy()

summary_group_columns = ["window_label", "test_months", "gap_months", "model"]
trend_shape_model_summary = (
    trend_shape_report.groupby(summary_group_columns, as_index=False)
    .agg(
        folds=("fold_id", "nunique"),
        mean_monthly_total_wmape_percent=("monthly_total_wmape_percent", "mean"),
        mean_rolling_3m_wmape_percent=("mean_rolling_3m_wmape_percent", "mean"),
        mean_actual_slope_per_month=("actual_slope_per_month", "mean"),
        mean_forecast_slope_per_month=("forecast_slope_per_month", "mean"),
        mean_abs_slope_gap_per_month=("slope_gap_per_month", lambda values: float(np.nanmean(np.abs(values)))),
        flat_forecast_fold_share_percent=("trend_status", lambda values: 100 * values.eq("flat_forecast_vs_actual_trend").mean()),
        mean_actual_forecast_correlation=("actual_forecast_correlation", "mean"),
    )
    .sort_values(["window_label", "mean_rolling_3m_wmape_percent", "model"])
    .reset_index(drop=True)
)

trend_shape_report.to_csv(table_dir / "lumpy_phase3_trend_shape_report.csv", index=False)
trend_shape_model_summary.to_csv(table_dir / "lumpy_phase3_trend_shape_model_summary.csv", index=False)
latest_required_trend.to_csv(table_dir / "lumpy_phase3_latest_required_trend_shape_report.csv", index=False)

print("Saved Phase 3 trend reports:")
print(table_dir / "lumpy_phase3_trend_shape_report.csv")
print(table_dir / "lumpy_phase3_trend_shape_model_summary.csv")
print(table_dir / "lumpy_phase3_latest_required_trend_shape_report.csv")

display(
    latest_required_trend.sort_values(["monthly_total_wmape_percent", "model"])[
        [
            "window_label",
            "model",
            "monthly_total_wmape_percent",
            "mean_rolling_3m_wmape_percent",
            "actual_slope_per_month",
            "forecast_slope_per_month",
            "slope_capture_ratio",
            "actual_forecast_correlation",
            "trend_status",
        ]
    ]
)

display(
    trend_shape_model_summary.loc[
        trend_shape_model_summary["test_months"].eq(REQUIRED_WINDOW_MONTHS)
        & trend_shape_model_summary["gap_months"].eq(REQUIRED_LAG_MONTHS)
    ]
)


## Recommendation


In [ ]:
lead_latest = latest_required_trend.loc[latest_required_trend["model"].eq(lead_model)]
if lead_latest.empty:
    print("No latest-fold trend row found for the lead model.")
else:
    row = lead_latest.iloc[0]
    print(f"Lead model: {lead_model}")
    print(f"Latest-fold monthly total WMAPE: {row['monthly_total_wmape_percent']:.1f}%")
    print(f"Latest-fold actual slope/month: {row['actual_slope_per_month']:.2f}")
    print(f"Latest-fold forecast slope/month: {row['forecast_slope_per_month']:.2f}")
    print(f"Trend status: {row['trend_status']}")

    if row["trend_status"] == "flat_forecast_vs_actual_trend":
        print(
            "Recommendation: keep the current model as the scoring baseline, but do not claim it captures trend. "
            "The next model experiment should target aggregate monthly shape first, then allocate back to SKUs."
        )
    elif row["trend_status"].startswith("trend_direction"):
        print("Recommendation: trend direction is present; focus next on SKU allocation and forecast bias.")
    else:
        print("Recommendation: inspect the latest fold before adding more model complexity.")


## Existing Agent Reports


In [ ]:
display_order = [
    "metric_decision",
    "monthly_total",
    "sku_horizon",
    "lag_comparison",
    "population_strategy",
    "segment_strategy",
]

for name in display_order:
    report = reports.get(name, pd.DataFrame())
    print(f"\n{name.replace('_', ' ').title()}")
    if report.empty:
        print("No report found.")
    else:
        display(report)
